# 🧵 DeepLure Saree AI — Kaggle Notebook
## Color-Invariant Design Identification via ArcFace Metric Learning

---

**Objective**: Identify saree designs independent of color palette — _face recognition for textiles_

**Architecture**:
- Backbone: `EfficientNet-B3` (ImageNet pretrained, ~10.7M params)
- Head: 512-d embedding + L2 normalization (unit hypersphere)
- Loss: `ArcFace` (additive angular margin m=0.5, scale s=64) — implemented from scratch
- Color Invariance: ColorJitter(hue=0.3, sat=0.8) + RandomGrayscale(p=0.3→0.7)

**Datasets Used**:
1. Indian Saree Patterns (Kaggle) — 1,468 images, 4 style classes [MIT License]
2. DeepLure Handloom Corpus — 165 images, no labels [Proprietary — not redistributed]

**Pretrained Weights**: EfficientNet-B3 from torchvision (ImageNet1K_V1) — disclosed

---

## 0. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm PyYAML

import sys, os
print(f'Python: {sys.version}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Clone repository (if running on Kaggle)
import os

REPO_URL = 'https://github.com/kamalds2/DeepLure_Saree_AI_Agent.git'
REPO_DIR = '/kaggle/working/DeepLure_Saree_AI_Agent'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## 1. Data Paths & Config

In [ ]:
import yaml

# ── EDIT THESE PATHS FOR YOUR KAGGLE SETUP ──────────────────────────────
# If using Kaggle dataset input:
KAGGLE_DATA_ROOT = '/kaggle/input/indian-saree-patterns'   # adjust to your dataset path
HANDLOOM_DIR     = '/kaggle/input/deeplure-handloom'       # adjust if available
# ─────────────────────────────────────────────────────────────────────────

# Update config with correct paths
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['kaggle_root']    = KAGGLE_DATA_ROOT
cfg['data']['handloom_root']  = HANDLOOM_DIR
cfg['paths']['checkpoint_dir'] = '/kaggle/working/checkpoints'
cfg['paths']['logs_dir']       = '/kaggle/working/logs'
cfg['paths']['reports_dir']    = '/kaggle/working/reports'

# Save updated config
with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config loaded:')
for k, v in cfg['data'].items():
    print(f'  {k}: {v}')

## 2. Dataset Exploration

In [ ]:
from data import KaggleSareeDataset, get_val_transforms, CLASS_TO_IDX
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import numpy as np

# Load training dataset (no transform for visualization)
train_ds = KaggleSareeDataset(KAGGLE_DATA_ROOT, split='train', transform=get_val_transforms(224))
val_ds   = KaggleSareeDataset(KAGGLE_DATA_ROOT, split='valid', transform=get_val_transforms(224))
test_ds  = KaggleSareeDataset(KAGGLE_DATA_ROOT, split='test',  transform=get_val_transforms(224))

print(f'\nDataset Summary:')
print(f'  Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
class_names = ['Banarasi', 'Bandhani', 'Ikat', 'Pichwai']

for row, cls_name in enumerate(class_names):
    cls_idx  = CLASS_TO_IDX[cls_name]
    cls_imgs = [p for p, l in train_ds.samples if l == cls_idx][:5]
    
    for col, img_path in enumerate(cls_imgs):
        ax = axes[row, col]
        img = Image.open(img_path).convert('RGB').resize((224, 224))
        ax.imshow(img)
        if col == 0:
            ax.set_ylabel(cls_name, fontsize=12, fontweight='bold', rotation=0, 
                         labelpad=60, va='center')
        ax.axis('off')

fig.suptitle('Sample Images per Style Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print('Sample images saved.')

## 3. Model Architecture

In [ ]:
import torch
from models import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Build model
model, criterion = build_model(cfg)
model     = model.to(device)
criterion = criterion.to(device)

# Test forward pass
dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    emb = model(dummy)
    
print(f'\nForward pass test:')
print(f'  Input shape:     {dummy.shape}')
print(f'  Output shape:    {emb.shape}')
print(f'  Embedding norms: {torch.norm(emb, dim=1).tolist()} (should be ~1.0)')

## 4. Training (3-Phase)

In [ ]:
# Run full 3-phase training
# Equivalent to: python train.py --config configs/config.yaml
import subprocess
result = subprocess.run(
    ['python', 'train.py', '--config', 'configs/config.yaml', '--device', 'cuda'],
    capture_output=False, text=True
)

# Alternatively, run inline:
# from train import main as train_main
# import argparse
# args = argparse.Namespace(config='configs/config.yaml', data_root=None,
#                           checkpoint_dir=None, device='cuda')
# train_main(args)

## 5. Training Curves

In [ ]:
from utils.visualization import plot_training_curves

log_path  = '/kaggle/working/logs/training_log.csv'
save_path = '/kaggle/working/training_curves.png'

if os.path.exists(log_path):
    plot_training_curves(log_path, save_path)
    img = Image.open(save_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print(f'Log file not found at {log_path}. Run training first.')

## 6. Build Gallery Index

In [ ]:
import subprocess

CHECKPOINT = '/kaggle/working/checkpoints/best_model.pth'
GALLERY_DIR = '/kaggle/working/gallery_index'

# Build gallery from Kaggle train+valid (labeled)
!python gallery_builder.py \
    --checkpoint {CHECKPOINT} \
    --gallery_source kaggle \
    --data_root {KAGGLE_DATA_ROOT} \
    --output_dir {GALLERY_DIR}

print('\nGallery index built!')
!ls -lh {GALLERY_DIR}

## 7. Evaluation: Identification + Verification

In [ ]:
!python evaluate.py \
    --checkpoint {CHECKPOINT} \
    --config configs/config.yaml \
    --mode kaggle

## 8. t-SNE Visualization

In [ ]:
import numpy as np
from gallery_builder import load_gallery_index
from utils.visualization import plot_tsne

# Load gallery embeddings
gallery_emb, gallery_labels, gallery_paths = load_gallery_index(GALLERY_DIR)

# Filter to labeled images only
mask = gallery_labels >= 0
emb_labeled = gallery_emb[mask]
lbl_labeled = gallery_labels[mask]

# Subsample for speed (t-SNE is O(N^2))
max_n = min(600, len(emb_labeled))
idx   = np.random.choice(len(emb_labeled), max_n, replace=False)

tsne_save = '/kaggle/working/tsne_gallery.png'
plot_tsne(
    emb_labeled[idx],
    lbl_labeled[idx],
    class_names=['Banarasi', 'Bandhani', 'Ikat', 'Pichwai'],
    save_path=tsne_save,
    title='t-SNE — ArcFace Embeddings (color = style class)',
)

img = Image.open(tsne_save)
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.title('t-SNE: Well-separated clusters → color-invariant texture discrimination')
plt.show()

## 9. Single Image Inference Demo

In [ ]:
# Run inference on a single test image
import random
test_sample_path, test_label = random.choice(test_ds.samples)

!python inference.py \
    --checkpoint {CHECKPOINT} \
    --gallery_dir {GALLERY_DIR} \
    --query {test_sample_path} \
    --top_k 5

## 10. Efficiency Report

In [ ]:
import time, torch
from models import build_model

model, criterion = build_model(cfg)
model = model.to(device)

# Parameter count
param_info = model.count_parameters()
arcface_params = sum(p.numel() for p in criterion.parameters())

print('═'*45)
print('  EFFICIENCY REPORT (Deliverable 4)')
print('═'*45)
print(f'  Backbone params:    {param_info["backbone"]:>12,}')
print(f'  Embedding head:     {param_info["head"]:>12,}')
print(f'  ArcFace (train):    {arcface_params:>12,}')
print(f'  Total:              {param_info["total"]:>12,}')
print(f'  Embedding size:     512-d × float32 = 2 KB/image')

# Inference latency
model.eval()
dummy = torch.randn(1, 3, 224, 224).to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy)

# Time 100 forward passes
torch.cuda.synchronize() if device.type == 'cuda' else None
t0 = time.perf_counter()
for _ in range(100):
    with torch.no_grad():
        _ = model(dummy)
torch.cuda.synchronize() if device.type == 'cuda' else None
elapsed = (time.perf_counter() - t0) / 100 * 1000

print(f'  Inference latency:  {elapsed:.1f} ms/image ({device})')

# FLOPs (approximate via input size)
try:
    from torchvision.models import efficientnet_b3
    # EfficientNet-B3: ~1.8B FLOPs for 224x224 (from EfficientNet paper)
    print(f'  FLOPs (224×224):    ~1.8 GFLOPs (EfficientNet-B3 standard)')
except:
    pass

# Gallery search time
gallery_emb = np.random.randn(1000, 512).astype(np.float32)
gallery_emb /= np.linalg.norm(gallery_emb, axis=1, keepdims=True)
query_emb_np = np.random.randn(512).astype(np.float32)
t0 = time.perf_counter()
for _ in range(1000):
    _ = gallery_emb @ query_emb_np
elapsed_search = (time.perf_counter() - t0) / 1000 * 1000
print(f'  Gallery search (N=1000): {elapsed_search:.3f} ms/query (CPU numpy)')
print('═'*45)

## 11. Color Invariance Demo
Show that same design in different colors → similar embeddings

In [ ]:
# Color invariance sanity check:
# Take one image, apply hue shifts, compare embeddings
import torchvision.transforms.functional as TF
from data import get_val_transforms

sample_path = test_ds.samples[0][0]
orig_img = Image.open(sample_path).convert('RGB')

# Create 4 color variants
hue_shifts = [0.0, 0.3, -0.3, 0.5]
variant_tensors = []
variant_imgs    = []

transform = get_val_transforms(224)
for h in hue_shifts:
    jittered = TF.adjust_hue(orig_img, h)
    variant_imgs.append(jittered)
    variant_tensors.append(transform(jittered))

batch = torch.stack(variant_tensors).to(device)
with torch.no_grad():
    embeddings = model(batch).cpu().numpy()

# Compute pairwise cosine similarities
sim_matrix = embeddings @ embeddings.T

print('Color Invariance Test — Pairwise Cosine Similarities:')
print('(Same image, different hue shifts — should all be HIGH > 0.7)')
print(f'Hue shifts tested: {hue_shifts}')
for i in range(4):
    for j in range(4):
        print(f'  hue={hue_shifts[i]:+.1f} vs hue={hue_shifts[j]:+.1f}: {sim_matrix[i,j]:.4f}')

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, h in zip(axes, variant_imgs, hue_shifts):
    ax.imshow(img)
    ax.set_title(f'Hue shift: {h:+.1f}', fontsize=10)
    ax.axis('off')
fig.suptitle('Color Invariance: Same Pattern, Different Palette', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/color_invariance_demo.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Summary

| Component | Details |
|---|---|
| **Backbone** | EfficientNet-B3 (ImageNet pretrained, ~10.7M params) |
| **Loss** | ArcFace (m=0.5, s=64) — implemented from scratch |
| **Embedding** | 512-d, L2-normalized (unit hypersphere) |
| **Color Invariance** | ColorJitter(hue=0.3, sat=0.8) + RandomGrayscale(p=0.3→0.7) |
| **Training** | 3-phase: warmup (5ep) → fine-tune (25ep) → color calibration (5ep) |
| **Identification** | Recall@1/5/10, mAP via cosine gallery search |
| **Verification** | ROC-AUC, EER, Accuracy at optimal threshold |
| **Inference** | < 20ms/image on Kaggle T4 GPU |

**External disclosures**:
- Pretrained weights: EfficientNet-B3 from torchvision (ImageNet)
- Data: Indian Saree Patterns from Kaggle (MIT)
- Data: DeepLure Handloom (proprietary — not in repo, deleted post-exercise)
- No 3rd-party metric learning libraries used
